# Cấp 5 – Huấn luyện Supervised Policy-Value Network trên Google Colab GPU T4

Tài liệu và mã nguồn phục vụ **Issue #10** và **Proposal mục 5** (Nhóm 31 - TTNTNC):
- **Kiến trúc:** ResNet nhỏ (6 khối residual, 64 kênh) hai đầu `policy` (121 logits, mask ô bận) + `value` ($[-1, 1]$ qua tanh).
- **Dữ liệu:** ~20.000 ván Alpha-Beta độ sâu 3 tự chơi, chuẩn hóa 8 phép đối xứng $D_4$ để loại trùng và chống rò rỉ thế cờ giữa train/val/test.
- **Hàm mất mát & Tối ưu:** Cross-Entropy (policy) + MSE (value) + L2 regularization; Adam lr 1e-3, batch size 256; early stopping theo validation loss.
- **Lưu trữ:** Tự động lưu checkpoint `best.pt`, `latest.pt` và log lên Google Drive (`MyDrive/caro/sl`) để không mất mát dữ liệu.

### Hướng dẫn chạy:
1. Vào **Runtime → Change runtime type → Chọn T4 GPU**.
2. Chạy lần lượt các ô bên dưới.

## 1. Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Tải mã nguồn và cài đặt thư viện

In [ ]:
%cd /content
!test -d TTNTNC-NHOM31 || git clone https://github.com/minhhieu310800-jpg/TTNTNC-NHOM31.git
%cd /content/TTNTNC-NHOM31
!git checkout main
!git pull --ff-only
%cd do-an-caro
!pip install -q -r requirements.txt

## 3. Kiểm tra phần cứng GPU (T4 GPU)

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:    ", torch.cuda.get_device_name(0))
    print("Memory (GB):    ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> T4 GPU.")

## 4. Chuẩn bị dữ liệu huấn luyện

Bạn có thể nạp file `dataset_sl.npz` từ Google Drive, hoặc sinh mới trực tiếp trên Colab nếu chưa có dữ liệu sẵn.

In [ ]:
import os, shutil
from pathlib import Path

# Thư mục dữ liệu trên Google Drive
DRIVE_DATA = Path('/content/drive/MyDrive/caro/data/dataset_sl.npz')
LOCAL_DATA = Path('data/dataset_sl.npz')
LOCAL_DATA.parent.mkdir(parents=True, exist_ok=True)

if DRIVE_DATA.exists():
    print(f"Tìm thấy dữ liệu trên Google Drive: {DRIVE_DATA}")
    print("Đang sao chép vào máy cục bộ để tối ưu tốc độ đọc SSD/RAM...")
    shutil.copy(DRIVE_DATA, LOCAL_DATA)
    print(f"Đã sao chép xong vào {LOCAL_DATA} ({LOCAL_DATA.stat().st_size / 1e6:.1f} MB).")
elif LOCAL_DATA.exists():
    print(f"Đã có sẵn dữ liệu cục bộ tại {LOCAL_DATA}.")
else:
    print("Chưa có dataset_sl.npz. Bắt đầu sinh 5.000 ván mẫu từ Alpha-Beta trên Colab...")
    !python -m training.generate_data configs/training/generate_data.yaml --limit 5000 --workers 4
    if LOCAL_DATA.exists():
        DRIVE_DATA.parent.mkdir(parents=True, exist_ok=True)
        print(f"Lưu trữ bản sao dữ liệu lên Google Drive: {DRIVE_DATA}")
        shutil.copy(LOCAL_DATA, DRIVE_DATA)

## 5. Huấn luyện Supervised Learning (Cấp 5)

- Sử dụng GPU T4 (`--device cuda`).
- Checkpoint `best.pt`, `latest.pt`, `train_log.jsonl`, và `summary.json` được lưu trực tiếp vào Google Drive để tránh mất mát khi phiên Colab bị ngắt.
- Nếu phiên bị ngắt quãng, thêm `--resume {OUT}/latest.pt` để tiếp tục chạy từ epoch dở dang.

In [ ]:
OUT = '/content/drive/MyDrive/caro/supervised-policy-value-network'
!python -m training.train_supervised_policy_value_network configs/training/supervised-policy-value-network.yaml \
    --data data/dataset_sl.npz \
    --device cuda \
    --out-dir {OUT} \
    --epochs 30 \
    --batch-size 256 \
    --patience 5

## 6. Vẽ biểu đồ học tập (Loss, Accuracy)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

log_file = Path(f"{OUT}/train_log.jsonl")
if log_file.exists():
    records = [json.loads(line) for line in log_file.read_text(encoding='utf-8').strip().split('\n') if line]
    epochs = [r['epoch'] for r in records]
    train_loss = [r['train']['loss'] for r in records]
    val_loss = [r['val']['loss'] for r in records]
    train_p_loss = [r['train']['policy_loss'] for r in records]
    val_p_loss = [r['val']['policy_loss'] for r in records]
    train_v_loss = [r['train']['value_loss'] for r in records]
    val_v_loss = [r['val']['value_loss'] for r in records]
    train_acc = [r['train']['top1_acc'] * 100 for r in records]
    val_acc = [r['val']['top1_acc'] * 100 for r in records]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Đồ thị Loss
    ax1.plot(epochs, train_loss, 'b-', label='Train Total Loss')
    ax1.plot(epochs, val_loss, 'r--', label='Val Total Loss')
    ax1.plot(epochs, val_p_loss, 'g:', label='Val Policy Loss')
    ax1.plot(epochs, val_v_loss, 'm:', label='Val Value Loss')
    ax1.set_title('Supervised Learning Loss Curves')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Đồ thị Accuracy
    ax2.plot(epochs, train_acc, 'b-', label='Train Policy Top-1 Acc (%)')
    ax2.plot(epochs, val_acc, 'r--', label='Val Policy Top-1 Acc (%)')
    ax2.set_title('Policy Top-1 Accuracy (%)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{OUT}/learning_curves.png", dpi=300)
    plt.show()
    print(f"Đã lưu biểu đồ học tập vào {OUT}/learning_curves.png")
else:
    print("Chưa có file train_log.jsonl để vẽ biểu đồ.")

## 7. Thử nghiệm Tác tử Cấp 5 (SLAgent) đấu thực chiến

Cho tác tử Cấp 5 sử dụng mô hình vừa huấn luyện (`best.pt`) chơi thử với HeuristicAgent hoặc RandomAgent.

In [ ]:
from game.game import CaroGame
from agents.supervised_policy_value_network_agent import SupervisedPolicyValueNetworkAgent as SLAgent
from agents.heuristic_agent import HeuristicAgent
from agents.random_agent import RandomAgent

best_checkpoint = f"{OUT}/best.pt"
sl_agent = SLAgent(checkpoint=best_checkpoint, device='cuda' if torch.cuda.is_available() else 'cpu')
opponent = HeuristicAgent()

game = CaroGame()
print("Bắt đầu ván đấu thử nghiệm: SLAgent (Đen/X) vs HeuristicAgent (Trắng/O)")

while not game.is_terminal():
    current_player = game.current_player
    if current_player == 1:  # Đen/X
        action = sl_agent.select_action(game)
        stats = sl_agent.last_stats
        r, c = divmod(action, 11)
        print(f"Nước {len(game.move_history)+1}: SLAgent đánh ô ({r}, {c}) [P={stats.get('policy_prob', 0):.3f}, V={stats.get('value', 0):.3f}]")
    else:
        action = opponent.select_action(game)
        r, c = divmod(action, 11)
        print(f"Nước {len(game.move_history)+1}: Heuristic đánh ô ({r}, {c})")
    game.step(action)

winner = game.result()
if winner == 1:
    print("\n=> KẾT QUẢ: SLAgent (Cấp 5) THẮNG!")
elif winner == -1:
    print("\n=> KẾT QUẢ: HeuristicAgent THẮNG!")
else:
    print("\n=> KẾT QUẢ: HÒA!")